In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from sklearn.model_selection import train_test_split

# Load annotations
train_annotations = pd.read_csv('data/processed/train/annotations.csv')
test_annotations = pd.read_csv('data/processed/test/annotations.csv')

# Display basic statistics
print("Train set stats:")
print(train_annotations.describe())
print("\nTest set stats:")
print(test_annotations.describe())

# Visualize class distribution
plt.figure(figsize=(10, 6))
sns.countplot(x='label', data=train_annotations)
plt.title('Class Distribution in Train Set')
plt.show()

plt.figure(figsize=(10, 6))
sns.countplot(x='label', data=test_annotations)
plt.title('Class Distribution in Test Set')
plt.show()

Explore video characteristics

In [ ]:
def get_video_info(video_path):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = frame_count / fps
    cap.release()
    return fps, frame_count, duration

train_fps = []
train_frame_count = []
train_duration = []

for video_path in train_annotations['video_path']:
    fps, frame_count, duration = get_video_info(video_path)
    train_fps.append(fps)
    train_frame_count.append(frame_count)
    train_duration.append(duration)

train_annotations['fps'] = train_fps
train_annotations['frame_count'] = train_frame_count
train_annotations['duration'] = train_duration

# Visualize video characteristics
fig, axes = plt.subplots(1, 3, figsize=(24, 6))

sns.boxplot(ax=axes[0], x='label', y='fps', data=train_annotations)
axes[0].set_title('FPS Distribution')

sns.boxplot(ax=axes[1], x='label', y='frame_count', data=train_annotations)
axes[1].set_title('Frame Count Distribution')

sns.boxplot(ax=axes[2], x='label', y='duration', data=train_annotations)
axes[2].set_title('Duration Distribution')

plt.tight_layout()
plt.show()

Sample video visualization

In [ ]:
def display_sample_video(video_path, num_frames=10):
    cap = cv2.VideoCapture(video_path)
    frames = []
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        if len(frames) < num_frames:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        else:
            break
    
    cap.release()
    
    fig, axes = plt.subplots(1, num_frames, figsize=(num_frames*4, 4))
    for i, ax in enumerate(axes):
        ax.imshow(frames[i])
        ax.axis('off')
    plt.show()

# Display a sample video from each class
real_video = train_annotations[train_annotations['label'] == 0]['video_path'].iloc[0]
fake_video = train_annotations[train_annotations['label'] == 1]['video_path'].iloc[0]

print("Sample Real Video:")
display_sample_video(real_video)

print("Sample Fake Video:")
display_sample_video(fake_video)

In [ ]:
Correlation analysis

corr_matrix = train_annotations[['fps', 'frame_count', 'duration']].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', square=True)
plt.title('Correlation Between Video Characteristics')
plt.show()